In [2]:
import pandas as pd
import os

# Carregar sua base já limpa
df = pd.read_csv('2025/base_senado_pronta_mineracao.csv', sep=';', encoding='utf-8')

# Agregar por departamento e mês para consolidar os gastos reais
df_depto = df.groupby(['LOTAÇÃO EXERCÍCIO', 'MES_REFERENCIA']).agg({
    'HORAS_EXTRAS': 'sum',
    'REMUN_BASICA': 'sum',
    'REM_LIQUIDA': 'sum',
    'FUNCAO_NUMERICA': 'mean', # Média hierárquica do setor
    'VÍNCULO': 'count'         # Quantidade de funcionários no setor
}).reset_index()

# Renomear coluna de contagem para representar o tamanho do departamento
df_depto.rename(columns={'VÍNCULO': 'QTD_SERVIDORES'}, inplace=True)

In [3]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Normalizar os dados (obrigatório para K-Means)
scaler = StandardScaler()
X = scaler.fit_transform(df_depto[['HORAS_EXTRAS', 'REMUN_BASICA', 'QTD_SERVIDORES']])

# Aplicar o K-Means (Exemplo com 3 grupos: Baixo Custo, Médio, Crítico)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_depto['CLUSTER_RISCO'] = kmeans.fit_predict(X)

In [12]:
# Criar categorias para o Apriori
zero_mask = df['HORAS_EXTRAS'] == 0
positive = df.loc[~zero_mask, 'HORAS_EXTRAS']

df.loc[zero_mask, 'FAIXA_HORA_EXTRA'] = 'Zero/Baixa'

if not positive.empty:
    df.loc[~zero_mask, 'FAIXA_HORA_EXTRA'] = pd.qcut(
        positive,
        q=[0, 0.7, 0.9, 1.0],
        labels=['Média', 'Alta', 'Muito_Alta'],
        duplicates='drop'
    )

df['PERFIL_FUNCAO'] = df['FUNCAO_NUMERICA'].map({0: 'Sem_Funcao', 1: 'Funcao_Baixa', 2: 'Funcao_Baixa', 3: 'Funcao_Media', 4: 'Funcao_Alta', 5: 'Funcao_Alta'})

In [13]:
! pip install mlxtend

  Using cached mlxtend-0.24.0-py3-none-any.whl.metadata (7.3 kB)
Using cached mlxtend-0.24.0-py3-none-any.whl (1.4 MB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 1.3 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/8.2 MB 1.5 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/8.2 MB 1.5 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/8.2 MB 1.6 MB/s eta 0:00:05
   ---------- ----------------------------- 2.1/8.2 MB 1.5 MB/s eta 0:00:04
   ----------- ---------------------------- 2.4/8.2 MB 1.5 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.2 MB 1.5 MB/s eta 0:00:04
   -------------- ------------------------- 2.9/8.2 MB 1.4 MB/s eta 0:00:04
   --------------- ------------------------ 3.1/8.2 MB 1.5 MB/s et

In [16]:
from mlxtend.frequent_patterns import apriori, association_rules

# Criar a matriz de transações (One-Hot Encoding das categorias)
df_associa = pd.get_dummies(df[['VÍNCULO', 'PERFIL_FUNCAO', 'FAIXA_HORA_EXTRA']])

# Encontrar itens frequentes e gerar as regras
itens_frequentes = apriori(df_associa, min_support=0.05, use_colnames=True)
regras = association_rules(itens_frequentes, metric="confidence", min_threshold=0.6)

# Filtrar regras onde a consequência é ter Hora Extra Alta
regras_criticas = regras[regras['consequents'].astype(str).str.contains('FAIXA_HORA_EXTRA_Alta')]